# **ORGANIZATION & NOTEBOOK INITIALIZATION (RUN EVERYTIME)**

In [ ]:
######################################################################
### IMPORT PACKAGES & SETUP NOTEBOOK (RUN AT BEGINNING EVERY TIME) ###
######################################################################

project_name = 'Phosphotriesterase_Wetlab_Data_Analysis'

### REPOSITORY ROOT (auto-detected -- nothing here needs editing) ###
# Walks up from the current directory looking for the repository markers, so
# this notebook works from a fresh clone anywhere on disk. Override by setting
# the ZINC_HYDRO_REPO environment variable if you have an unusual layout.
def _find_repo_root():
    import os as _os
    from pathlib import Path as _Path
    _markers = ('Scripts', 'Software', 'Environment', 'LICENSE')
    _start = _Path(_os.environ.get('ZINC_HYDRO_REPO') or _Path.cwd()).resolve()
    for _cand in (_start, *_start.parents):
        if all((_cand / _m).exists() for _m in _markers):
            return str(_cand)
    raise RuntimeError(
        f"Could not find the repository root above {_start}. "
        f"Start Jupyter from inside the cloned repository, or set ZINC_HYDRO_REPO."
    )

github_repo_dir = _find_repo_root()
# NOTE: this notebook needs no external tools -- only numpy/pandas/scipy/
#       matplotlib/openpyxl. Open Babel belongs to the DESIGN pipelines.

### STANDARD LIBRARY & THIRD-PARTY IMPORTS ###
import glob, json, math, os, re, sys, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import AutoMinorLocator
from IPython.display import display

### PLATE-READER PARSING & FITTING ###
# Vendored under Scripts/wetlab_platereader/ so this notebook is self-contained.
_platereader = os.path.join(github_repo_dir, 'Scripts', 'wetlab_platereader')
if _platereader not in sys.path:
    sys.path.insert(0, _platereader)
from kinetics import parse_standard_curve, parse_kinetics, plot_kinetics, export_plot
from neo2_util import parse_neo2_kinetics

### DIRECTORIES ###
notebook_dir        = os.path.join(github_repo_dir, 'Manuscript_Data',
                                   'Phosphotriesterase_RFdiffusion3')
raw_wetlab_data_dir = os.path.join(notebook_dir, 'raw_wetlab_data')
wetlab_data_plots_dir = os.path.join(notebook_dir, 'wetlab_data_plots')
os.makedirs(wetlab_data_plots_dir, exist_ok=True)

def raw(prefix):
    """Absolute path to the one raw data file whose name starts with `prefix`.

    The instrument writes long descriptive filenames; matching on a short
    distinctive prefix keeps the cells below readable and fails loudly rather
    than silently picking the wrong plate.
    """
    hits = sorted(glob.glob(os.path.join(raw_wetlab_data_dir, prefix + '*')))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly one raw file starting with {prefix!r}, found {len(hits)}"
            + (":\n  " + "\n  ".join(os.path.basename(h) for h in hits) if hits else ""))
    return hits[0]

### FIGURE OUTPUT (applies to every figure this notebook writes) ###
# All figures are written to wetlab_data_plots/ as PNG. PNG at a modest DPI
# keeps the repository small -- these are the files committed to GitHub.
# EPS is vector and much larger, so it is off by default; flip SAVE_EPS to True
# (or pass save_eps=True at a single call site) when you need one for a figure.
FIGURE_DPI = 150     # raster resolution for the PNGs
SAVE_EPS   = False   # also write <name>.eps alongside <name>.png

def save_figure(name, fig=None, dpi=None, save_eps=None):
    """Write a figure to wetlab_data_plots/ as PNG (and optionally EPS).

    Pass `fig` explicitly wherever you can. The plotting helpers call plt.show()
    before returning, and under Jupyter's inline backend that closes the figure,
    so a later plt.gcf() would hand back a fresh empty one and save a blank PNG.
    """
    stem = os.path.join(wetlab_data_plots_dir, name)
    export_plot(f'{stem}.png', fig=fig, dpi=FIGURE_DPI if dpi is None else dpi, transparent=False)
    if SAVE_EPS if save_eps is None else save_eps:
        export_plot(f'{stem}.eps', fig=fig, transparent=False)
    return f'{stem}.png'

### CALIBRATION CONSTANTS (derived in section I.I; hard-coded here so every ###
### kinetics cell below can be run on its own without re-running section I) ###
SPU_NOBIC = 0.00633035   # signal per uM 4-nitrophenol, legacy buffer, no NaHCO3
SPU_BIC   = 0.00667808   # signal per uM 4-nitrophenol, legacy buffer + 25 mM NaHCO3

### FITTED KINETICS ARE COLLECTED HERE AND SUMMARIZED IN SECTION III ###
FITS = {}

### OPTIONALS ###
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=RuntimeWarning)

### PRINTS ###
print(f"### PROJECT {project_name} NOTEBOOK SUCCESSFULLY INITIALIZED ON {datetime.now():%Y-%m-%d %H:%M} ###\n")
print(f"repository root : {github_repo_dir}")
print(f"raw data        : {raw_wetlab_data_dir}  ({len(os.listdir(raw_wetlab_data_dir))} files)")
print(f"figures         : {wetlab_data_plots_dir}  (PNG @ {FIGURE_DPI} dpi, EPS {'on' if SAVE_EPS else 'off'})")

# **I. CALIBRATION & BACKGROUND REACTION RATE**

## I.I. 4-Nitrophenol Standard Curves

Paraoxon hydrolysis is followed at 405 nm, where the product 4-nitrophenol
absorbs. Converting absorbance to product concentration needs a calibration
slope measured in the same buffer as the reaction, because 4-nitrophenol's
extinction coefficient depends on pH and the phenolate/phenol ratio shifts
with buffer composition.

One plate covers four conditions in triplicate: two buffers ("legacy" and
"modern") each with and without 25 mM NaHCO<sub>3</sub>, over an eight-point
serial dilution from 200 µM. The kinetics below were all run in legacy buffer,
so they use the first two slopes.

In [ ]:
######################################################
### 4-NITROPHENOL STANDARD CURVES (FOUR CONDITIONS) ##
######################################################

### INPUT ###
standard_curve_file = raw('260225_StdCurve_200uMserials_vert__80uL')
CONC_STD = [200, 100, 50, 25, 12.5, 6.25, 3.175, 1.5875]   # uM, serial dilution down the rows

### CONDITIONS (three replicate columns each) ###
STD_CONDITIONS = [
    ("legacy buffer, no NaHCO3",        [1, 2, 3]),
    ("legacy buffer + 25 mM NaHCO3",    [4, 5, 6]),
    ("modern buffer, no NaHCO3",        [7, 8, 9]),
    ("modern buffer + 25 mM NaHCO3",    [10, 11, 12]),
]

### EXECUTION ###
print(f"{'condition':32} {'columns':14} slope (signal per uM)")
print("-" * 70)
standard_curve_slopes = {}
for name, cols in STD_CONDITIONS:
    slope, _ = parse_standard_curve(
        data_path=standard_curve_file,
        blocks=[dict(rows=list("ABCDEFGH"), concentrations_uM=CONC_STD, replicate_cols=cols)],
        force_zero_intercept=False, plot=False)
    standard_curve_slopes[name] = slope
    print(f"{name:32} {str(cols):14} {slope:.8f}")

print(f"\nThe two legacy-buffer slopes are the ones used below:")
print(f"  SPU_NOBIC = {SPU_NOBIC:.8f}   SPU_BIC = {SPU_BIC:.8f}")

## I.II. Uncatalyzed Rate for Paraoxon Hydrolysis (*k*<sub>uncat</sub>)

Enzyme-free paraoxon hydrolysis, 0.3–9.6 mM, followed for 4.5 h, with and
without 25 mM NaHCO<sub>3</sub>.

*k*<sub>uncat</sub> is the slope of v<sub>0</sub> against [S] fitted with a
**free intercept**, which absorbs substrate-independent baseline drift.
Subtracting that intercept from every v<sub>0</sub> and refitting through the
origin returns the same slope, so the free-intercept fit is used directly.

Rather than choosing one fit window by hand, every admissible window is scanned
(at least 45 min long, R<sup>2</sup> ≥ 0.90, on a 5-min grid) and the reported
value is the median across them. The uncertainty combines the 16th–84th
percentile spread across windows with the median within-window regression
error, so it reflects both the choice of window and the fit itself.

In [ ]:
##########################################################
### UNCATALYZED PARAOXON HYDROLYSIS (k_uncat), +/- HCO3 ##
##########################################################

### INPUT ###
kuncat_file = raw('260807_kuncat_legacy_buffer')
ROWS_K = list("ABCDEF")                                   # one row per [paraoxon]
CONC_K = np.array([9600, 4800, 2400, 1200, 600, 300], float)   # uM
KCFG = {"no NaHCO3":     dict(cols=[1, 2, 3], spu=SPU_NOBIC),
        "+25 mM NaHCO3": dict(cols=[4, 5, 6], spu=SPU_BIC)}

### LOAD AND RESHAPE THE TIME COURSE ###
_raw_kuncat = parse_neo2_kinetics(kuncat_file)
_d = _raw_kuncat.groupby(["Well", "time"], as_index=False).agg(signal=("value", "mean"))
_d["time"] = _d["time"].astype(float) - _d.groupby("Well")["time"].transform("min")
KPIV = _d.pivot(index="time", columns="Well", values="signal").sort_index()
KT = KPIV.index.to_numpy(float)

### FITTING ###
def kuncat_block(condition, lo, hi):
    """Mean v0 per [S] over one fit window, plus the free-intercept fit."""
    cfg = KCFG[condition]
    m = (KT >= lo) & (KT <= hi)
    tt = KT[m]
    reps = [np.polyfit(tt, np.column_stack([KPIV[f"{r}{c}"].to_numpy(float)[m]
            for c in cfg["cols"]]), 1)[0] / cfg["spu"] for r in ROWS_K]
    V = np.array([x.mean() for x in reps])
    E = np.array([x.std(ddof=1) / np.sqrt(len(x)) for x in reps])
    k, c = np.polyfit(CONC_K, V, 1)
    pred = k * CONC_K + c
    ss = np.sum((V - V.mean()) ** 2)
    return k, c, (1 - np.sum((V - pred) ** 2) / ss if ss > 0 else np.nan), V, E

def kuncat_report(condition, grid=300., min_duration=2700., min_start=900.):
    """Median k_uncat over every admissible fit window, with combined error."""
    accepted = []
    for lo in np.arange(min_start, KT.max() - min_duration + 1, grid):
        for hi in np.arange(lo + min_duration, KT.max() + 1, grid):
            k, c, r2, V, E = kuncat_block(condition, lo, hi)
            if r2 >= 0.90:
                # leave-one-out spread across the six substrate concentrations
                jk = np.array([np.polyfit(np.delete(CONC_K, i), np.delete(V, i), 1)[0]
                               for i in range(len(CONC_K))])
                accepted.append((lo, hi, k, np.sqrt(5/6 * np.sum((jk - jk.mean()) ** 2))))
    A = np.array(accepted)
    k = A[:, 2]
    median = np.median(k)
    p16, p84 = np.percentile(k, [16, 84])
    stat = np.median(A[:, 3])
    err = max(np.hypot(median - p16, stat), np.hypot(p84 - median, stat))
    best = A[np.argmin(np.abs(k - median))]        # the window closest to the median
    return dict(k=median, err=err, n=len(A), lo=best[0], hi=best[1])

### EXECUTION ###
KUNCAT = {c: kuncat_report(c) for c in KCFG}
for condition, r in KUNCAT.items():
    half_life_lo = np.log(2) / (r["k"] + r["err"]) / 86400
    half_life_hi = np.log(2) / (r["k"] - r["err"]) / 86400
    print(f"{condition:16} k_uncat = ({r['k']*1e8:.1f} +/- {r['err']*1e8:.1f}) x 1e-8 s^-1"
          f"   t_1/2 = {half_life_lo:.0f}-{half_life_hi:.0f} d   ({r['n']} windows)")

# **II. MICHAELIS-MENTEN KINETICS**

## II.I. Shared Fitting Routine

Every design below was measured the same way, so they all go through one
routine. A plate holds three enzyme replicate columns and one background column
per design, with paraoxon serially diluted down rows A–F (9.6 mM to 0.3 mM).

For each design the routine fits an initial rate to every enzyme replicate and
every background replicate independently, subtracts the per-row mean background
slope from each enzyme replicate, converts signal to product with the
condition-matched calibration slope, and fits Michaelis–Menten to the mean
rates. Fits are collected in `FITS` and summarized in section III.

`time_range_seconds` is the window over which initial rates are taken. It is
shorter for the plates where product accumulates fastest.

In [ ]:
##################################################
### SHARED MICHAELIS-MENTEN FITTING ROUTINE    ###
##################################################

### STANDARD PLATE LAYOUT ###
KINETICS_ROWS   = list("ABCDEF")
KINETICS_CONC   = [9600, 4800, 2400, 1200, 600, 300]   # uM paraoxon, down the rows

def run_kinetics(name, *, file_prefix, enzyme_uM, enzyme_cols, bg_cols,
                 condition="+25 mM NaHCO3", time_range_seconds=(300, 20000),
                 plot_name=None, scaffold=None, plate_id=None, show=True):
    """Fit one design's Michaelis-Menten kinetics, plot it, and record the result.

    Returns the fit dict from parse_kinetics and stores it in FITS[name].
    """
    signal_per_uM = SPU_BIC if condition == "+25 mM NaHCO3" else SPU_NOBIC
    df, reps_df, mm = parse_kinetics(
        data_path=raw(file_prefix),
        blocks=[dict(rows=KINETICS_ROWS, concentrations_uM=KINETICS_CONC,
                     enzyme_cols=list(enzyme_cols), bg_cols=list(bg_cols))],
        enzyme_uM=enzyme_uM,
        signal_per_uM=signal_per_uM,
        time_range_seconds=time_range_seconds,
        baseline_subtract=True,
    )
    if show:
        fig = plot_kinetics(df, reps_df, mm, time_scale="minute")
        if plot_name:
            save_figure(plot_name, fig=fig)
    FITS[name] = dict(mm=mm, condition=condition, enzyme_uM=enzyme_uM,
                      scaffold=scaffold, plate_id=plate_id)
    return mm

## II.II. Round 1 Designs (PET_i1 Scaffold)

The five designs carried forward from the first design round, all on the same
scaffold. ZAPP-1 was additionally measured without bicarbonate as a control on
the buffer condition; that run uses the no-bicarbonate calibration slope.

### II.II.A. ZAPP-1 (p1D1)

[E]<sub>0</sub> = 24.0 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-1',
    file_prefix = '260804_ZAPP1strp_24uM',
    enzyme_uM   = 24.0,
    enzyme_cols = [1, 2, 3], bg_cols = [4],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D1',
    plot_name   = 'kinetics_ZAPP1_p1D1',
)

### II.II.B. ZAPP-1 (p1D1), without bicarbonate

[E]<sub>0</sub> = 37.3 µM, paraoxon 0.3–9.6 mM, no NaHCO₃, 1 % MeOH, 405 nm.

Buffer control. Fitted with `SPU_NOBIC`, the calibration slope measured in the same buffer, so it is directly comparable to the run above.

In [ ]:
run_kinetics(
    'ZAPP-1 (no NaHCO3)',
    file_prefix = '260225_p1D1_30pt9uM',
    enzyme_uM   = 37.3,
    enzyme_cols = [1, 2, 3], bg_cols = [4],
    condition   = 'no NaHCO3',
    scaffold    = 'PET_i1',
    plate_id    = 'p1D1',
    plot_name   = 'kinetics_ZAPP1_p1D1_no_bicarbonate',
)

### II.II.C. R1 p1D7

[E]<sub>0</sub> = 21.3 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'R1 p1D7',
    file_prefix = '260224_p1D7_21pt3uM',
    enzyme_uM   = 21.3,
    enzyme_cols = [1, 2, 3], bg_cols = [4],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D7',
    plot_name   = 'kinetics_R1_p1D7',
)

### II.II.D. R1 p1D8

[E]<sub>0</sub> = 26.1 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'R1 p1D8',
    file_prefix = '260224_p1D1_26pt2uM',
    enzyme_uM   = 26.1,
    enzyme_cols = [5, 6, 7], bg_cols = [8],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D8',
    plot_name   = 'kinetics_R1_p1D8',
)

### II.II.E. R1 p1E10

[E]<sub>0</sub> = 15.3 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'R1 p1E10',
    file_prefix = '260224_p1D7_21pt3uM',
    enzyme_uM   = 15.3,
    enzyme_cols = [5, 6, 7], bg_cols = [8],
    scaffold    = 'PET_i1',
    plate_id    = 'p1E10',
    plot_name   = 'kinetics_R1_p1E10',
)

## II.III. Round 2 Designs (PTE_i2 Scaffolds)

Seven designs from the second round, spanning four scaffolds (s1, s3, s6, s7).
Where product accumulates fastest the initial-rate window is shortened to
300–660 s so the fit stays in the linear region of the progress curve.

### II.III.A. ZAPP-2 (p2C4), scaffold 3

[E]<sub>0</sub> = 26.6 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-2',
    file_prefix = '260731_p1F6_34pt8uM',
    enzyme_uM   = 26.6,
    enzyme_cols = [5, 6, 7], bg_cols = [8],
    time_range_seconds = (300, 660),
    scaffold    = 'PTE_i2_s3',
    plate_id    = 'p2C4',
    plot_name   = 'kinetics_ZAPP2_p2C4',
)

### II.III.B. R2 p2B12, scaffold 3

[E]<sub>0</sub> = 10.3 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'R2 p2B12',
    file_prefix = '260731_p1F6_34pt8uM',
    enzyme_uM   = 10.3,
    enzyme_cols = [9, 10, 11], bg_cols = [12],
    time_range_seconds = (300, 660),
    scaffold    = 'PTE_i2_s3',
    plate_id    = 'p2B12',
    plot_name   = 'kinetics_R2_p2B12',
)

### II.III.C. ZAPP-3 (p2G1), scaffold 7

[E]<sub>0</sub> = 27.5 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-3',
    file_prefix = '260731_p2E3_28pt9uM',
    enzyme_uM   = 27.5,
    enzyme_cols = [5, 6, 7], bg_cols = [8],
    scaffold    = 'PTE_i2_s7',
    plate_id    = 'p2G1',
    plot_name   = 'kinetics_ZAPP3_p2G1',
)

### II.III.D. ZAPP-4 (p2E3), scaffold 6

[E]<sub>0</sub> = 28.9 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-4',
    file_prefix = '260731_p2E3_28pt9uM',
    enzyme_uM   = 28.9,
    enzyme_cols = [1, 2, 3], bg_cols = [4],
    scaffold    = 'PTE_i2_s6',
    plate_id    = 'p2E3',
    plot_name   = 'kinetics_ZAPP4_p2E3',
)

### II.III.E. ZAPP-5 (p1F6), scaffold 1

[E]<sub>0</sub> = 34.8 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-5',
    file_prefix = '260731_p1F6_34pt8uM',
    enzyme_uM   = 34.8,
    enzyme_cols = [1, 2, 3], bg_cols = [4],
    time_range_seconds = (300, 660),
    scaffold    = 'PTE_i2_s1',
    plate_id    = 'p1F6',
    plot_name   = 'kinetics_ZAPP5_p1F6',
)

### II.III.F. R2 p1D9, scaffold 1

[E]<sub>0</sub> = 44.6 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'R2 p1D9',
    file_prefix = '260804_ZAPP1strp_24uM',
    enzyme_uM   = 44.6,
    enzyme_cols = [5, 6, 7], bg_cols = [8],
    scaffold    = 'PTE_i2_s1',
    plate_id    = 'p1D9',
    plot_name   = 'kinetics_R2_p1D9',
)

### II.III.G. R2 p1H4, scaffold 1

[E]<sub>0</sub> = 39.1 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'R2 p1H4',
    file_prefix = '260804_ZAPP1strp_24uM',
    enzyme_uM   = 39.1,
    enzyme_cols = [9, 10, 11], bg_cols = [12],
    scaffold    = 'PTE_i2_s1',
    plate_id    = 'p1H4',
    plot_name   = 'kinetics_R2_p1H4',
)

## II.IV. ZAPP-1 Active-Site Mutants

Six single-site mutants of ZAPP-1, measured under the same conditions as the
parent so the fits are directly comparable.

### II.IV.A. ZAPP-1 MUT1

[E]<sub>0</sub> = 32.7 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-1 MUT1',
    file_prefix = '260804_p2H3_4pt6uM',
    enzyme_uM   = 32.7,
    enzyme_cols = [5, 6, 7], bg_cols = [8],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D1',
    plot_name   = 'kinetics_ZAPP1_MUT1',
)

### II.IV.B. ZAPP-1 MUT2

[E]<sub>0</sub> = 25.8 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-1 MUT2',
    file_prefix = '260804_p2H3_4pt6uM',
    enzyme_uM   = 25.8,
    enzyme_cols = [9, 10, 11], bg_cols = [12],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D1',
    plot_name   = 'kinetics_ZAPP1_MUT2',
)

### II.IV.C. ZAPP-1 MUT3

[E]<sub>0</sub> = 28.1 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-1 MUT3',
    file_prefix = '260804_ZAPPmut3_28pt1uM',
    enzyme_uM   = 28.1,
    enzyme_cols = [1, 2, 3], bg_cols = [4],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D1',
    plot_name   = 'kinetics_ZAPP1_MUT3',
)

### II.IV.D. ZAPP-1 MUT4

[E]<sub>0</sub> = 18.7 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-1 MUT4',
    file_prefix = '260804_ZAPPmut3_28pt1uM',
    enzyme_uM   = 18.7,
    enzyme_cols = [5, 6, 7], bg_cols = [8],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D1',
    plot_name   = 'kinetics_ZAPP1_MUT4',
)

### II.IV.E. ZAPP-1 MUT5

[E]<sub>0</sub> = 39.5 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-1 MUT5',
    file_prefix = '260804_ZAPPmut3_28pt1uM',
    enzyme_uM   = 39.5,
    enzyme_cols = [9, 10, 11], bg_cols = [12],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D1',
    plot_name   = 'kinetics_ZAPP1_MUT5',
)

### II.IV.F. ZAPP-1 MUT6

[E]<sub>0</sub> = 23.2 µM, paraoxon 0.3–9.6 mM, 25 mM NaHCO₃, 1 % MeOH, 405 nm.

In [ ]:
run_kinetics(
    'ZAPP-1 MUT6',
    file_prefix = '260804_ZAPPmut6_23pt2uM',
    enzyme_uM   = 23.2,
    enzyme_cols = [1, 2, 3], bg_cols = [4],
    scaffold    = 'PET_i1',
    plate_id    = 'p1D1',
    plot_name   = 'kinetics_ZAPP1_MUT6',
)

# **III. SUMMARY OF FITTED KINETIC PARAMETERS**

## III.I. Combined Table

Every fit recorded in section II, with the condition-matched
*k*<sub>uncat</sub> from section I.II and the derived ratios. Errors on the
ratios propagate the relative errors of the two terms in quadrature.

The table is assembled from the fits held in `FITS`, so it always reflects the
cells actually run above rather than a transcribed copy. Run section II in full
before this cell.

In [ ]:
############################################
### SUMMARY TABLE OF ALL FITTED DESIGNS  ###
############################################

### CHECK EVERYTHING UPSTREAM HAS BEEN RUN ###
if not FITS:
    raise RuntimeError("FITS is empty -- run the kinetics cells in section II first.")
if 'KUNCAT' not in globals():
    raise RuntimeError("KUNCAT is not defined -- run section I.II first.")

### ASSEMBLE ###
rows = []
for name, f in FITS.items():
    mm = f['mm']
    rows.append({
        "design":            name,
        "plate ID":          f.get('plate_id'),
        "scaffold":          f.get('scaffold'),
        "condition":         f['condition'],
        "[E]0 (uM)":         f['enzyme_uM'],
        "kcat (s-1)":        mm['kcat_per_s'],
        "kcat sd":           mm['kcat_err_per_s'],
        "Km (uM)":           mm['Km_uM'],
        "Km sd":             mm['Km_err_uM'],
        "kcat/Km (M-1 s-1)": mm['kcat_over_Km_per_uM_per_s'] * 1e6,
        "kcat/Km sd":        mm['kcat_over_Km_err_per_uM_per_s'] * 1e6,
    })
summary_df = pd.DataFrame(rows)

### CONDITION-MATCHED UNCATALYZED RATE ###
summary_df["kuncat (s-1)"]  = summary_df["condition"].map(lambda c: KUNCAT[c]["k"])
summary_df["kuncat sd"]     = summary_df["condition"].map(lambda c: KUNCAT[c]["err"])

### DERIVED RATIOS (relative errors added in quadrature) ###
def _rel(a, sa, b, sb):
    return np.hypot(sa / a, sb / b)

summary_df["kcat/kuncat"] = summary_df["kcat (s-1)"] / summary_df["kuncat (s-1)"]
summary_df["kcat/kuncat sd"] = summary_df["kcat/kuncat"] * _rel(
    summary_df["kcat (s-1)"], summary_df["kcat sd"],
    summary_df["kuncat (s-1)"], summary_df["kuncat sd"])
summary_df["(kcat/Km)/kuncat (M-1)"] = summary_df["kcat/Km (M-1 s-1)"] / summary_df["kuncat (s-1)"]
summary_df["(kcat/Km)/kuncat sd"] = summary_df["(kcat/Km)/kuncat (M-1)"] * _rel(
    summary_df["kcat/Km (M-1 s-1)"], summary_df["kcat/Km sd"],
    summary_df["kuncat (s-1)"], summary_df["kuncat sd"])

### WRITE ###
summary_csv = os.path.join(wetlab_data_plots_dir, "kinetics_summary.csv")
summary_df.to_csv(summary_csv, index=False, float_format="%.6g")
print(f"{len(summary_df)} fits -> {summary_csv}\n")

### DISPLAY ###
_show = ["design", "scaffold", "condition", "[E]0 (uM)",
         "kcat (s-1)", "kcat sd", "Km (uM)", "kcat/Km (M-1 s-1)", "kcat/Km sd"]
summary_df[_show].style.format({
    "kcat (s-1)": "{:.5f}", "kcat sd": "{:.5f}", "Km (uM)": "{:.0f}",
    "kcat/Km (M-1 s-1)": "{:.3f}", "kcat/Km sd": "{:.3f}"})

# **IV. ELUATE SCREENING**

## IV.I. Round 1 Eluate Screen (96-Well)

Purified eluate from each design in the first order, one well per design,
followed at 405 nm. 300 µM paraoxon, 1 % MeOH, 200 µM ZnSO<sub>4</sub>,
HEPES elution buffer, 1 h at room temperature.

Each trace is normalized to its own first reading so wells are compared on
change in absorbance rather than absolute signal, which varies with eluate
concentration. The fastest-climbing wells are labeled.

In [ ]:
###################################################
### ROUND 1 ELUATE SCREEN - PROGRESS CURVES     ###
###################################################

### PROGRESS-CURVE HELPERS ###
# Vendored alongside the kinetics parsers; see Scripts/wetlab_platereader/.
import screening_curves
screening_curves.SAVE_DPI = FIGURE_DPI          # keep it in step with this notebook
from screening_curves import draw_progression_curve_v0

### INPUT ###
screen_r1_file = raw('250919_order1_plate1')
rows, columns  = list("ABCDEFGH"), list(range(1, 13))

### PARAMETERS ###
hit_definition_num = 4        # how many of the fastest wells to label
blank              = []       # wells to treat as blank (none on this plate)
ignore             = []       # wells to leave out entirely
time_start, time_end = 0, 12  # minutes

### OUTPUT ###
screen_r1_plot = os.path.join(wetlab_data_plots_dir, 'screen_round1_progress_curves.png')

### EXECUTION ###
draw_progression_curve_v0(
    screen_r1_file, rows, columns, hit_definition_num, blank,
    time_start, time_end, label=True, start_row=36, ignore=ignore,
    save_path=screen_r1_plot,
)
print(f"saved {screen_r1_plot}")

## IV.II. Round 2 Eluate Screen (384-Well)

The second order, 192 designs across two 96-well source plates, stamped into a
single 384-well reader plate and read for an hour. 600 µM paraoxon, 1 % MeOH,
50 mM bicarbonate buffer, 60 µL reactions, 405 nm, 25 °C.

Because two source plates are interleaved into one reader plate, the mapping
from reader well back to design is the step most worth checking; IV.II.B tests
it against the designs that were independently sequenced.

### IV.II.A. Assay Constants and Well Mapping

Reader wells are mapped back to their source plate and 96-well position, then
joined to the ordered-design table built from the order FASTA headers. The
mapping carries its own assertions, so a mis-stamped plate fails here rather
than silently mislabeling every design downstream.

In [ ]:
##########################################
### ROUND 2 SCREEN - ASSAY CONSTANTS   ###
##########################################

### PLATE / ASSAY ###
EXPECTED_N_DESIGNS      = 192
EXPECTED_N_READER_WELLS = 192
SUBSTRATE_UM       = 600.0     # paraoxon
RXN_VOLUME_UL      = 60.0
COSOLVENT          = '1% (v/v) MeOH'
BUFFER             = '50 mM bicarbonate ("legacy" buffer)'
READ_WAVELENGTH_NM = 405
ASSAY_TEMP_C       = 25

### INITIAL-VELOCITY FIT WINDOW (minutes) ###
FIT_START_MIN = 2.0
FIT_END_MIN   = 15.0

### QC THRESHOLDS ###
ABS_LINEAR_CEILING = 2.0       # absorbance is only trusted below this
DROP_ABS_TOL       = 0.010     # a single-read decrease beyond this is an optical artifact
MIN_FIT_POINTS     = 8

### HIT CALLING ###
HIT_Z_THRESHOLD    = 5.0       # robust z above the inactive population
HIT_FOLD_THRESHOLD = 3.0       # and at least this many fold over background

### FIGURE SIZING (Nature: 89 mm single column, 183 mm double) ###
MM = 1 / 25.4
WIDTH_1COL = 89 * MM
WIDTH_2COL = 183 * MM

### PLOT PALETTE ###
# Okabe-Ito hues, chosen to stay distinguishable under deuteranopia and protanopia.
SCAFFOLD_COLORS = {
    1:  '#0072B2',   # blue
    2:  '#56B4E9',   # sky blue
    3:  '#D55E00',   # vermillion
    4:  '#E69F00',   # orange
    5:  '#A6761D',   # dark gold
    6:  '#009E73',   # bluish green
    7:  '#CC79A7',   # reddish purple
    8:  '#5D3A9B',   # violet
    9:  '#66A61E',   # olive
    10: '#A65628',   # brown
    11: '#E7298A',   # magenta
}
BACKGROUND_TRACE_COLOR = '#b8b8b8'   # inactive progress curves
FIT_WINDOW_COLOR       = '#f2c94c'   # shaded initial-rate window

def style_science_axes(ax, title=None, x_label=None, y_label=None,
                       title_size=7.5, label_size=7, tick_size=6, spine_width=0.6):
    """Boxed, inward-ticked axes - the house style, scaled to publication point sizes."""
    if title is not None:
        ax.set_title(title, fontsize=title_size, weight='semibold')
    if x_label is not None:
        ax.set_xlabel(x_label, fontsize=label_size, weight='semibold')
    if y_label is not None:
        ax.set_ylabel(y_label, fontsize=label_size, weight='semibold')
    ax.tick_params(axis='both', which='major', labelsize=tick_size,
                   direction='in', length=2.6, width=spine_width, top=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in', length=1.5,
                   width=spine_width * 0.83, top=True, right=True)
    for side in ['left', 'right', 'top', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(spine_width)
        ax.spines[side].set_edgecolor('black')

print(f"{EXPECTED_N_DESIGNS} designs, {SUBSTRATE_UM:g} uM paraoxon, {BUFFER}, "
      f"{READ_WAVELENGTH_NM} nm, {ASSAY_TEMP_C} C")
print(f"initial rates fitted over {FIT_START_MIN:g}-{FIT_END_MIN:g} min")

In [ ]:
####################################################################
### MAP READER WELLS TO ORDERED DESIGNS AND SCAFFOLDS            ###
####################################################################

### INPUTS ###
platereader_csv      = raw('260525_i3smw')
order_fasta          = raw('260514__ARuder')
design_decisions_csv = raw('design_decisions')

expected_n_designs      = EXPECTED_N_DESIGNS
expected_n_reader_wells = EXPECTED_N_READER_WELLS

#####################
### HELPER LOGIC  ###
#####################

DESIGN_NAME_RE = re.compile(
    r'(?P<design_id>ZAPP_i3_AR_pte_(?P<design_index>\d+))'
    r'_scaffold_(?P<scaffold>\d+)'
    r'____Plate(?P<plate_number>\d)__(?P<well_position>[A-H]\d{1,2})'
    r'____(?P<parent>.+?)(?:\.pdb)?$'
)

def parse_design_name(name):
    """Pull design id, scaffold and plate position out of an ordered design name."""
    match = DESIGN_NAME_RE.match(str(name).strip())
    if match is None:
        raise ValueError(f'Could not parse ordered design name: {name}')
    fields = match.groupdict()
    return {
        'design_id':     fields['design_id'],
        'design_index':  int(fields['design_index']),
        'scaffold':      int(fields['scaffold']),
        'plate_number':  int(fields['plate_number']),
        'well_position': fields['well_position'].upper(),
        'parent':        fields['parent'],
    }

def load_design_table(fasta_path):
    """Build the ordered-design table from the FASTA headers sent to the vendor."""
    headers = [line[1:].strip() for line in Path(fasta_path).read_text().splitlines()
               if line.startswith('>')]
    if len(headers) != expected_n_designs:
        raise ValueError(f'Expected {expected_n_designs} FASTA records, found {len(headers)}.')

    design_df = pd.DataFrame([parse_design_name(h) for h in headers])
    design_df['source_well'] = ('P' + design_df['plate_number'].astype(str)
                                + design_df['well_position'])
    # the ORI_* token identifies the diffusion parent that defines each scaffold family
    design_df['ori_family'] = design_df['parent'].str.extract(r'(ORI_\d+_C\d+_i_\d+_model_\d+)')

    if design_df['design_id'].duplicated().any():
        raise ValueError('Duplicate design IDs in the order FASTA.')
    if design_df.duplicated(['plate_number', 'well_position']).any():
        raise ValueError('Duplicate plate/well positions in the order FASTA.')
    if not design_df['plate_number'].isin([1, 2]).all():
        raise ValueError('Found a plate number outside {1, 2}.')
    if not design_df['well_position'].str.fullmatch(r'[A-H](?:[1-9]|1[0-2])').all():
        raise ValueError('Found an invalid 96-well position.')

    # each scaffold must correspond to exactly one diffusion parent
    family_counts = design_df.groupby('scaffold')['ori_family'].nunique()
    if (family_counts != 1).any():
        raise ValueError('A scaffold maps to more than one ORI parent family.')
    return design_df

def map_reader_well_to_plate_well(reader_well):
    """384-well reader position -> (source plate number, 96-well position)."""
    match = re.fullmatch(r'([A-H])(\d{1,2})', str(reader_well))
    if match is None:
        raise ValueError(f'Invalid reader well label: {reader_well}')

    reader_row = match.group(1)
    reader_col = int(match.group(2))
    if reader_col < 1 or reader_col > 24:
        raise ValueError(f'Reader well column outside 1-24: {reader_well}')

    reader_row_idx = 'ABCDEFGH'.index(reader_row)
    plate_number = 1 if reader_row_idx < 4 else 2
    row_pairs = [('A', 'B'), ('C', 'D'), ('E', 'F'), ('G', 'H')]
    row_pair = row_pairs[reader_row_idx % 4]
    well_row = row_pair[0] if reader_col % 2 == 1 else row_pair[1]
    well_col = (reader_col + 1) // 2
    return plate_number, f'{well_row}{well_col}'

def get_reader_well_columns(df):
    well_cols = [col for col in df.columns if re.fullmatch(r'[A-H](?:[1-9]|1\d|2[0-4])', str(col))]
    if len(well_cols) != expected_n_reader_wells:
        raise ValueError(f'Expected {expected_n_reader_wells} reader wells, found {len(well_cols)}.')
    return well_cols

def build_reader_map(well_cols):
    """Reader-well -> plate/well table, with the mapping's own unit tests."""
    example_expectations = {
        'A1':  (1, 'A1'),
        'A2':  (1, 'B1'),
        'B5':  (1, 'C3'),
        'E4':  (2, 'B2'),
        'H24': (2, 'H12'),
    }
    for reader_well, expected in example_expectations.items():
        observed = map_reader_well_to_plate_well(reader_well)
        if observed != expected:
            raise AssertionError(f'{reader_well} mapped to {observed}, expected {expected}.')

    reader_map_df = pd.DataFrame([
        {'reader_well': w, **dict(zip(['plate_number', 'well_position'],
                                      map_reader_well_to_plate_well(w)))}
        for w in well_cols
    ])
    if reader_map_df.duplicated(['plate_number', 'well_position']).any():
        raise ValueError('Reader-well mapping produced duplicate 96-well positions.')
    return reader_map_df

def load_reader_traces(platereader_csv):
    """Return (time_min, raw absorbance frame indexed by reader well)."""
    reader_df = pd.read_csv(platereader_csv)
    well_cols = get_reader_well_columns(reader_df)

    raw_time = pd.to_numeric(reader_df['Time'], errors='coerce')
    if raw_time.isna().any():
        raise ValueError('Plate-reader Time column contains non-numeric values.')
    # the reader writes elapsed time as a fraction of a day
    time_min = (raw_time - raw_time.iloc[0]) * 24 * 60

    abs_df = reader_df[well_cols].apply(pd.to_numeric, errors='coerce')
    if abs_df.isna().any().any():
        raise ValueError('Plate-reader absorbance block contains non-numeric values.')
    return time_min.to_numpy(dtype=float), abs_df, well_cols

#################
### LOAD DATA ###
#################

design_df = load_design_table(order_fasta)
time_min, abs_df, reader_well_cols = load_reader_traces(platereader_csv)
reader_map_df = build_reader_map(reader_well_cols)

well_df = reader_map_df.merge(design_df, on=['plate_number', 'well_position'],
                              how='left', validate='one_to_one')
if well_df['design_id'].isna().any():
    display(well_df[well_df['design_id'].isna()])
    raise ValueError('Some reader wells did not receive a design assignment.')

n_timepoints = len(time_min)
read_interval_min = float(np.median(np.diff(time_min)))

print(f'Reader wells mapped:  {len(well_df)}')
print(f'Designs assigned:     {well_df["design_id"].nunique()}')
print(f'Scaffolds present:    {sorted(well_df["scaffold"].unique())}')
print(f'Timepoints:           {n_timepoints} ({time_min[0]:.2f}-{time_min[-1]:.2f} min, '
      f'{read_interval_min:.2f} min interval)')
print(f'Absorbance range:     {abs_df.values.min():.3f}-{abs_df.values.max():.3f} '
      f'A{READ_WAVELENGTH_NM}')
print()
print('Designs per scaffold:')
display(well_df['scaffold'].value_counts().sort_index().rename('n_designs').to_frame().T)

### IV.II.B. Mapping Validation

Two stamping hypotheses are compared against the designs that were
independently sequenced: the interleaved row-pair mapping used above, and a
plain column-halves alternative. The mapping that puts the sequenced designs
in the right wells is the correct one. Skipped automatically if the sequencing
table is absent.

In [ ]:
####################################################################
### VALIDATE THE READER MAPPING AGAINST THE SEQUENCED DESIGNS    ###
####################################################################

### INPUTS ###
validation_fit_start_min = FIT_START_MIN
validation_fit_end_min   = FIT_END_MIN

#####################
### HELPER LOGIC  ###
#####################

def quick_ols_rate(y, t, start_min, end_min):
    """Plain OLS slope over a time window - used only for mapping validation."""
    mask = (t >= start_min) & (t <= end_min)
    return float(np.polyfit(t[mask], y[mask], 1)[0])

def column_halves_mapping(reader_well):
    """Alternative stamping hypothesis: plate 1 = columns 1-12, plate 2 = columns 13-24."""
    row, col = re.fullmatch(r'([A-H])(\d+)', reader_well).groups()
    col = int(col)
    plate = 1 if col <= 12 else 2
    return f'P{plate}{row}{col if col <= 12 else col - 12}'

#################
### LOAD DATA ###
#################

if not Path(design_decisions_csv).exists():
    print(f'Sequencing decisions not found, skipping validation: {design_decisions_csv}')
else:
    decisions_df = pd.read_csv(design_decisions_csv)
    sequenced_wells = sorted(set(decisions_df['design']))

    provisional_rate = pd.Series(
        {w: quick_ols_rate(abs_df[w].to_numpy(float), time_min,
                           validation_fit_start_min, validation_fit_end_min)
         for w in reader_well_cols}
    )

    validation_rows = []
    for label, source_lookup in [
        ('interleaved row-pairs (used here)',
         dict(zip(well_df['reader_well'], well_df['source_well']))),
        ('column halves (alternative)',
         {w: column_halves_mapping(w) for w in reader_well_cols}),
    ]:
        ranked = (pd.DataFrame({'reader_well': list(source_lookup),
                                'source_well': list(source_lookup.values())})
                  .assign(rate=lambda d: provisional_rate[d['reader_well']].to_numpy())
                  .assign(rank=lambda d: d['rate'].rank(ascending=False).astype(int))
                  .set_index('source_well'))
        ranks = ranked.loc[sequenced_wells, 'rank'].sort_values()
        validation_rows.append({
            'mapping': label,
            'ranks_of_sequenced_designs': list(ranks.values),
            'n_in_top_10': int((ranks <= 10).sum()),
            'n_in_top_20': int((ranks <= 20).sum()),
        })

    validation_df = pd.DataFrame(validation_rows)
    print(f'Designs sent for sequencing (n={len(sequenced_wells)}): {sequenced_wells}')
    display(validation_df)

    n_top = validation_df.loc[0, 'n_in_top_10']
    # probability of >= n_top of 8 randomly chosen designs landing in the top 10 of 192
    p_chance = stats.hypergeom.sf(n_top - 1, expected_n_designs, 10, len(sequenced_wells))
    print(f'Interleaved mapping puts {n_top}/{len(sequenced_wells)} sequenced designs in the '
          f'top 10 (p = {p_chance:.2e} by chance).')
    if validation_df.loc[0, 'n_in_top_20'] <= validation_df.loc[1, 'n_in_top_20']:
        raise AssertionError('The interleaved mapping is not better supported than the alternative.')
    print('Mapping confirmed.')

### IV.II.C. Initial Rates, QC Flags and Hit Calling

An initial rate is fitted per well over the window set above, after excluding
reads above the absorbance ceiling and wells showing optical artifacts. Hits
are called on a robust z-score against the inactive population combined with a
fold-over-background threshold, so both criteria have to agree.

In [ ]:
####################################################################
### FIT INITIAL RATES, FLAG ARTIFACTS AND CALL HITS              ###
####################################################################

### INPUTS ###
fit_start_min      = FIT_START_MIN
fit_end_min        = FIT_END_MIN
abs_linear_ceiling = ABS_LINEAR_CEILING
drop_abs_tol       = DROP_ABS_TOL
min_fit_points     = MIN_FIT_POINTS
hit_z_threshold    = HIT_Z_THRESHOLD
hit_fold_threshold = HIT_FOLD_THRESHOLD

### OUTPUTS ###
rates_csv = os.path.join(wetlab_data_plots_dir, 'round2_screen_initial_rates.csv')

#####################
### HELPER LOGIC  ###
#####################

def robust_sd(values):
    """MAD-based standard deviation estimate."""
    values = np.asarray(values, dtype=float)
    return 1.4826 * float(np.median(np.abs(values - np.median(values))))

def find_drop_artifacts(y, tol):
    """Indices of reads where absorbance falls by more than `tol` in a single step."""
    return np.flatnonzero(np.diff(y) < -tol) + 1

def longest_clean_segment(n_points, break_indices, window_mask):
    """Longest run of in-window reads uninterrupted by an artifact."""
    bounds = [0, *sorted(break_indices), n_points]
    best = None
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        seg = np.zeros(n_points, dtype=bool)
        seg[lo:hi] = True
        seg &= window_mask
        if best is None or seg.sum() > best.sum():
            best = seg
    return best

def fit_initial_rate(y, t):
    """OLS initial rate with artifact-aware segment selection.

    Returns the slope in absorbance units per minute plus the diagnostics needed to
    decide whether to trust it.
    """
    window_mask = (t >= fit_start_min) & (t <= fit_end_min)
    drops = find_drop_artifacts(y, drop_abs_tol)

    fit_mask = window_mask
    segmented = False
    if len(drops) and window_mask[drops].any():
        fit_mask = longest_clean_segment(len(y), drops, window_mask)
        segmented = True

    if fit_mask.sum() < min_fit_points:
        # fall back to the full window rather than fitting a handful of reads
        fit_mask = window_mask
        segmented = False

    x, yy = t[fit_mask], y[fit_mask]
    slope, intercept = np.polyfit(x, yy, 1)
    predicted = slope * x + intercept
    ss_res = float(np.sum((yy - predicted) ** 2))
    ss_tot = float(np.sum((yy - yy.mean()) ** 2))

    return {
        'rate_abs_per_min':  float(slope),
        'fit_intercept_abs': float(intercept),
        'fit_r2':            np.nan if ss_tot == 0 else 1 - ss_res / ss_tot,
        'fit_residual_sd':   float(np.sqrt(ss_res / max(len(x) - 2, 1))),
        'fit_n_points':      int(fit_mask.sum()),
        'fit_first_min':     float(x.min()),
        'fit_last_min':      float(x.max()),
        'fit_segmented':     segmented,
        'initial_abs':       float(y[0]),
        'final_abs':         float(y[-1]),
        'max_abs':           float(y.max()),
        'max_abs_in_window': float(y[window_mask].max()),
        'n_drop_artifacts':  int(len(drops)),
        'first_drop_min':    float(t[drops[0]]) if len(drops) else np.nan,
        'largest_drop_abs':  float(np.diff(y).min()),
    }

#################
### LOAD DATA ###
#################

rate_records = []
for reader_well in reader_well_cols:
    y = abs_df[reader_well].to_numpy(dtype=float)
    rate_records.append({'reader_well': reader_well, **fit_initial_rate(y, time_min)})

rates_df = well_df.merge(pd.DataFrame(rate_records), on='reader_well',
                         how='left', validate='one_to_one')

# >> QC flags
rates_df['flag_optical_artifact'] = rates_df['n_drop_artifacts'] > 0
rates_df['flag_above_linear_range'] = rates_df['max_abs'] > abs_linear_ceiling
rates_df['flag_saturated_in_window'] = rates_df['max_abs_in_window'] > abs_linear_ceiling
rates_df['qc_pass'] = ~(rates_df['flag_optical_artifact'] | rates_df['flag_saturated_in_window'])

# >> background: iteratively trim the active tail to isolate the inactive population
inactive = rates_df.loc[rates_df['qc_pass'], 'rate_abs_per_min'].to_numpy(float)
for _ in range(50):
    center, spread = np.median(inactive), robust_sd(inactive)
    keep = np.abs(inactive - center) <= 3 * spread
    if keep.all():
        break
    inactive = inactive[keep]

background_rate = float(np.median(inactive))
background_sd   = robust_sd(inactive)
n_inactive      = int(len(inactive))

rates_df['rate_corrected_abs_per_min'] = rates_df['rate_abs_per_min'] - background_rate
rates_df['robust_z'] = (rates_df['rate_abs_per_min'] - background_rate) / background_sd
rates_df['fold_over_background'] = rates_df['rate_abs_per_min'] / background_rate

rates_df['is_hit'] = (
    (rates_df['robust_z'] >= hit_z_threshold) &
    (rates_df['fold_over_background'] >= hit_fold_threshold) &
    rates_df['qc_pass']
)
rates_df['activity_rank'] = rates_df['rate_abs_per_min'].rank(ascending=False, method='min').astype(int)
rates_df = rates_df.sort_values('activity_rank').reset_index(drop=True)

limit_of_detection = 3 * background_sd

Path(wetlab_data_plots_dir).mkdir(parents=True, exist_ok=True)
rates_df.to_csv(rates_csv, index=False)

print(f'Fit window:                 {fit_start_min:g}-{fit_end_min:g} min')
print(f'Wells refitted on a segment: {int(rates_df["fit_segmented"].sum())}')
print(f'QC failures:                 {int((~rates_df["qc_pass"]).sum())} '
      f'({int(rates_df["flag_optical_artifact"].sum())} optical artifact, '
      f'{int(rates_df["flag_saturated_in_window"].sum())} above linear range in window)')
print(f'Wells exceeding A{READ_WAVELENGTH_NM} {abs_linear_ceiling:g} at any time: '
      f'{int(rates_df["flag_above_linear_range"].sum())}')
print()
print(f'Inactive population:         n = {n_inactive}')
print(f'Background rate:             {background_rate:.3e} +/- {background_sd:.3e} '
      f'A{READ_WAVELENGTH_NM}/min')
print(f'Limit of detection (3 SD):   {limit_of_detection:.3e} A{READ_WAVELENGTH_NM}/min')
print(f'Dynamic range (max/bkg):     {rates_df["rate_abs_per_min"].max() / background_rate:.0f}-fold')
print()
print(f'Hits (z >= {hit_z_threshold:g} and >= {hit_fold_threshold:g}x background): '
      f'{int(rates_df["is_hit"].sum())} / {len(rates_df)} '
      f'({100 * rates_df["is_hit"].mean():.1f}%)')
print(f'Saved: {rates_csv}')
print()

display_cols = ['activity_rank', 'reader_well', 'source_well', 'design_id', 'scaffold',
                'rate_abs_per_min', 'robust_z', 'fold_over_background', 'fit_r2',
                'max_abs', 'qc_pass', 'is_hit']
print('Top 15 designs by initial rate:')
display(rates_df.head(15)[display_cols])

### IV.II.D. Progress Curves by Scaffold

Every well's trace, colored by scaffold, with the fastest designs highlighted
and the fit window shaded.

In [ ]:
####################################################################
### PROGRESS CURVES COLORED BY DESIGN SCAFFOLD                   ###
####################################################################

### INPUTS ###
n_highlight = 8
screening_basename  = 'round2_screen_progress_curves'
manuscript_basename = 'round2_screen_progress_curves_compact'

### PLOT CONTROLS ###
figure_size          = (WIDTH_1COL * 1.35, WIDTH_1COL * 1.0)
background_linewidth = 0.35
background_alpha     = 0.55
highlight_linewidth  = 1.1

# manuscript display names for the hit-bearing scaffolds
MANUSCRIPT_SCAFFOLD_NAMES = {
    1: 'ZAPP-2 Scaffold',
    3: 'ZAPP-3 Scaffold',
    6: 'ZAPP-4 Scaffold',
    7: 'ZAPP-5 Scaffold',
}

#####################
### HELPER LOGIC  ###
#####################

def blank_normalized_trace(reader_well):
    """Absorbance relative to that well's first read."""
    y = abs_df[reader_well].to_numpy(dtype=float)
    return y - y[0]

# highlight the eight designs whose blank-normalized curve reaches the highest
# absorbance. No QC filtering: raw screening plates are polyclonal and unpurified.
rates_df['peak_delta_abs'] = [float(blank_normalized_trace(w).max())
                              for w in rates_df['reader_well']]
highlight_df = rates_df.nlargest(n_highlight, 'peak_delta_abs')
highlight_wells = set(highlight_df['reader_well'])
qc_fail_wells = set(rates_df.loc[~rates_df['qc_pass'], 'reader_well'])  # used by later cells

# blank-normalized absorbance at which the reader leaves its linear range
linear_ceiling_delta = ABS_LINEAR_CEILING - float(rates_df['initial_abs'].median())

#################
### PLOTTING  ###
#################

def plot_progress_curves(manuscript):
    fig, ax = plt.subplots(figsize=figure_size)

    if not manuscript:
        ax.axvspan(FIT_START_MIN, FIT_END_MIN, color=FIT_WINDOW_COLOR, alpha=0.16,
                   linewidth=0, zorder=0)

    # >> every non-highlighted well as a light gray background trace
    for reader_well in reader_well_cols:
        if reader_well in highlight_wells:
            continue
        ax.plot(time_min, blank_normalized_trace(reader_well), color=BACKGROUND_TRACE_COLOR,
                linewidth=background_linewidth, alpha=background_alpha,
                solid_capstyle='round', zorder=1)

    # >> highlighted designs, colored by scaffold (drawn lowest-peak first)
    for _, row in highlight_df.sort_values('peak_delta_abs').iterrows():
        ax.plot(time_min, blank_normalized_trace(row['reader_well']),
                color=SCAFFOLD_COLORS[row['scaffold']], linewidth=highlight_linewidth,
                solid_capstyle='round', zorder=3)

    style_science_axes(ax, x_label='Time (min)',
                       y_label=f'$\\Delta A_{{{READ_WAVELENGTH_NM}}}$ (blank-normalized)')
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.set_xlim(0, time_min[-1])

    scaffolds_shown = sorted(highlight_df['scaffold'].unique())

    if manuscript:
        # start at zero so the negative-going (bubble) traces fall off the bottom
        ax.set_ylim(bottom=0)
        handles = [Line2D([], [], color=SCAFFOLD_COLORS[s], linewidth=highlight_linewidth,
                          label=MANUSCRIPT_SCAFFOLD_NAMES.get(s, f'scaffold {s}'))
                   for s in scaffolds_shown]
        ax.legend(handles=handles, loc='upper left', fontsize=6, handlelength=1.6,
                  labelspacing=0.35, borderpad=0.3)
    else:
        # >> the fastest wells leave the reader's reliable range late in the run
        ax.axhline(linear_ceiling_delta, color='#555555', linewidth=0.5,
                   linestyle=(0, (4, 2)), zorder=4)
        ax.annotate(f'detector linear limit ($A_{{{READ_WAVELENGTH_NM}}}\\approx{ABS_LINEAR_CEILING:g}$)',
                    xy=(0.5, linear_ceiling_delta), xytext=(0, 2), textcoords='offset points',
                    ha='left', va='bottom', fontsize=5, color='#555555')
        # >> label the three highest-climbing designs directly
        for _, row in highlight_df.nlargest(3, 'peak_delta_abs').iterrows():
            trace = blank_normalized_trace(row['reader_well'])
            ax.annotate(row['design_id'].replace('ZAPP_i3_AR_pte_', 'i3-'),
                        xy=(time_min[-1], trace[-1]), xytext=(-2, 0), textcoords='offset points',
                        ha='right', va='center', fontsize=5.5,
                        color=SCAFFOLD_COLORS[row['scaffold']], fontweight='bold')
        handles = [Line2D([], [], color=SCAFFOLD_COLORS[s], linewidth=highlight_linewidth,
                          label=f'scaffold {s} (n={int((rates_df["scaffold"] == s).sum())})')
                   for s in scaffolds_shown]
        handles += [Line2D([], [], color=BACKGROUND_TRACE_COLOR, linewidth=1.0, label='other designs'),
                    Patch(facecolor=FIT_WINDOW_COLOR, alpha=0.16,
                          label=f'fit window ({FIT_START_MIN:g}-{FIT_END_MIN:g} min)')]
        ax.legend(handles=handles, loc='upper left', fontsize=5.5, handlelength=1.6,
                  labelspacing=0.35, borderpad=0.3)

    fig.tight_layout()
    return fig

save_figure(screening_basename, fig=plot_progress_curves(manuscript=False))
plt.show()
save_figure(manuscript_basename, fig=plot_progress_curves(manuscript=True))
plt.show()

print(f'Highlighted the {len(highlight_df)} highest-climbing designs, '
      f'spanning scaffolds {sorted(highlight_df["scaffold"].unique())}:')
display(highlight_df[['reader_well', 'source_well', 'design_id', 'scaffold',
                      'peak_delta_abs', 'rate_abs_per_min']]
        .rename(columns={'peak_delta_abs': 'peak_deltaA405'}))

### IV.II.E. Hit Table

In [ ]:
####################################################################
### EXPORT THE HIT TABLE                                          ###
####################################################################

### INPUTS ###
hits_csv = os.path.join(wetlab_data_plots_dir, 'round2_screen_hits.csv')

#################
### EXPORT    ###
#################

export_cols = [
    'activity_rank', 'design_id', 'scaffold', 'plate_number', 'well_position',
    'source_well', 'reader_well',
    'rate_abs_per_min', 'rate_corrected_abs_per_min', 'rate_um_per_min',
    'robust_z', 'fold_over_background', 'fit_r2', 'fit_residual_sd', 'fit_n_points',
    'initial_abs', 'final_abs', 'max_abs',
    'qc_pass', 'flag_optical_artifact', 'flag_above_linear_range', 'fit_segmented',
    'is_hit', 'parent',
]
export_cols = [c for c in export_cols if c in rates_df.columns]

hits_df = rates_df[rates_df['is_hit']].sort_values('activity_rank')
hits_df[export_cols].to_csv(hits_csv, index=False)

print(f'Screen summary')
print(f'  designs screened      : {len(rates_df)}')
print(f'  QC passing            : {int(rates_df["qc_pass"].sum())}')
print(f'  hits                  : {len(hits_df)} ({100 * len(hits_df) / len(rates_df):.1f}%)')
print(f'  scaffolds with a hit  : {sorted(hits_df["scaffold"].unique())}')
print(f'  best design           : {hits_df.iloc[0]["design_id"]} '
      f'(scaffold {hits_df.iloc[0]["scaffold"]}, '
      f'{hits_df.iloc[0]["fold_over_background"]:.0f}x background)')
print(f'  saved                 : {hits_csv}')
print()
print('CAVEATS FOR THE METHODS SECTION')
print('  - n = 1 per design; the two source plates hold 192 distinct designs, not replicates.')
print('  - rates are volumetric activities of unnormalized IMAC eluate, not specific')
print('    activities, so expression level and activity are confounded.')
print('  - absolute molar rates depend on an uncalibrated pathlength and buffer pH.')
print('  - no dedicated positive/negative control wells, so no Z-factor is reportable.')
print()
display(hits_df[export_cols].head(25))